# 02 – Build Master Dataset
Reads raw files, cleans each source, merges into quarterly and annual panels.

**Outputs:** `data/processed/quarterly_master.csv`, `data/processed/annual_net_additions.csv`

In [3]:
import pandas as pd
import numpy as np
import os

In [4]:
# ── 1. STARTS & COMPLETIONS (quarterly, England) ────────────────
df_sc = pd.read_excel(
    "../data/raw/ons_starts_completions_england.xlsx",
    sheet_name="1b",
    skiprows=5
)
df_sc = df_sc.dropna(subset=["Period", "Started - All Dwellings"])

df_sc["year"] = df_sc["Period"].str.extract(r"(\d{4})").astype(int)
df_sc["quarter"] = df_sc["Period"].apply(
    lambda x: 1 if "Jan" in str(x) else 2 if "Apr" in str(x) else 3 if "Jul" in str(x) else 4
)
df_sc["date"] = pd.to_datetime(
    df_sc["year"].astype(str) + "-" + (df_sc["quarter"] * 3 - 2).astype(str) + "-01"
)

df_sc = df_sc[["date", "year", "quarter",
               "Started - All Dwellings", "Started - Private Enterprise",
               "Completed - All Dwellings", "Completed - Private Enterprise"]].copy()
df_sc.columns = ["date", "year", "quarter",
                 "starts_all", "starts_private", "comp_all", "comp_private"]

print(f"Starts/completions: {df_sc.shape[0]} quarters, {df_sc.date.min().date()} to {df_sc.date.max().date()}")

Starts/completions: 191 quarters, 1978-01-01 to 2025-07-01


In [5]:
# ── 2. OBR HOUSING MARKET (quarterly, UK) ───────────────────────
df_obr = pd.read_excel(
    "../data/raw/obr_economy_march2026.xlsx",
    sheet_name="1.16",
    skiprows=1
)
df_obr = df_obr.iloc[:, 1:]  # drop empty first column

df_obr.columns = ["quarter_label", "hpi", "hpi_yoy", "transactions",
                   "starts_uk", "comp_uk", "housing_stock",
                   "net_additions_uk", "turnover_rate"]
df_obr = df_obr.iloc[1:].copy()  # drop header row

# Keep only quarterly rows (e.g. '2008Q1'), drop annual/fiscal-year/footnotes
mask = df_obr["quarter_label"].astype(str).str.match(r"^\d{4}Q\d$")
df_obr = df_obr[mask].copy()

df_obr["year"] = df_obr["quarter_label"].str[:4].astype(int)
df_obr["quarter"] = df_obr["quarter_label"].str[-1].astype(int)
df_obr["date"] = pd.to_datetime(
    df_obr["year"].astype(str) + "-" + (df_obr["quarter"] * 3 - 2).astype(str) + "-01"
)

num_cols = ["hpi", "hpi_yoy", "transactions", "starts_uk", "comp_uk",
            "housing_stock", "net_additions_uk", "turnover_rate"]
df_obr[num_cols] = df_obr[num_cols].apply(pd.to_numeric, errors="coerce")
df_obr = df_obr[["date", "year", "quarter"] + num_cols].reset_index(drop=True)

print(f"OBR housing market: {df_obr.shape[0]} quarters, {df_obr.date.min().date()} to {df_obr.date.max().date()}")
print(f"  Note: includes OBR forecast quarters from 2025 onwards")

OBR housing market: 93 quarters, 2008-01-01 to 2031-01-01
  Note: includes OBR forecast quarters from 2025 onwards


In [6]:
# ── 3. NET ADDITIONS (annual, England) ──────────────────────────
df_na = pd.read_excel(
    "../data/raw/mhclg_net_additions_england.ods",
    sheet_name="LT120_unrounded",
    engine="odf"
)

# Years across columns, components down rows — extract what we need
years_raw = df_na.iloc[3, 1:-2].tolist()
years = [str(y).split(" ")[0] for y in years_raw]
labels = df_na.iloc[:, 0].tolist()

rows_we_want = {
    "new_build_comp":      "New build completions",
    "net_conversions":     "Net conversions",
    "net_change_of_use":   "Net change of use",
    "net_other_gains":     "Net other gains",
    "demolitions":         "Demolitions",
    "total_net_additions": "Total net additional dwellings",
}

data = {"fiscal_year": years}
for col_name, label in rows_we_want.items():
    row_idx = labels.index(label)
    values = df_na.iloc[row_idx, 1:-2].tolist()
    data[col_name] = pd.to_numeric(values, errors="coerce")

df_net = pd.DataFrame(data)
df_net["year"] = df_net["fiscal_year"].str[:4].astype(int)

print(f"Net additions: {df_net.shape[0]} years, {df_net.year.min()} to {df_net.year.max()}")
print(f"  Demolitions fell from {df_net.demolitions.iloc[0]:.0f} (2006-07) to {df_net.demolitions.iloc[-1]:.0f} (2024-25)")

Net additions: 19 years, 2006 to 2024
  Demolitions fell from 22290 (2006-07) to 4632 (2024-25)


In [7]:
# ── 4. EPC NEW BUILDS (quarterly, England) ──────────────────────
df_epc = pd.read_excel(
    "../data/raw/epc_new_builds.ods",
    sheet_name="NB1_England_Only",
    engine="odf"
)

df_epc.columns = df_epc.iloc[2].tolist()
df_epc = df_epc.iloc[3:].copy()
df_epc = df_epc.dropna(subset=["Quarter"]).copy()  # keep quarterly rows only

df_epc["year"] = df_epc["Year"].astype(int)
df_epc["quarter"] = df_epc["Quarter"].str.split("/").str[1].astype(int)
df_epc["date"] = pd.to_datetime(
    df_epc["year"].astype(str) + "-" + (df_epc["quarter"] * 3 - 2).astype(str) + "-01"
)

df_epc = df_epc[["date", "year", "quarter", "Number Lodgements"]].copy()
df_epc.columns = ["date", "year", "quarter", "epc_lodgements"]
df_epc["epc_lodgements"] = pd.to_numeric(df_epc["epc_lodgements"], errors="coerce")
df_epc = df_epc.reset_index(drop=True)

print(f"EPC lodgements: {df_epc.shape[0]} quarters, {df_epc.date.min().date()} to {df_epc.date.max().date()}")

EPC lodgements: 69 quarters, 2008-10-01 to 2025-10-01


In [8]:
# ── 5. MERGE INTO QUARTERLY MASTER ──────────────────────────────
df_master = df_sc.copy()
df_master = df_master.merge(df_obr, on=["date", "year", "quarter"], how="left")
df_master = df_master.merge(df_epc, on=["date", "year", "quarter"], how="left")

print(f"Quarterly master: {df_master.shape}")
print(f"Date range: {df_master.date.min().date()} to {df_master.date.max().date()}")
print(f"\nNon-null counts:")
print(df_master.notna().sum())

Quarterly master: (191, 16)
Date range: 1978-01-01 to 2025-07-01

Non-null counts:
date                191
year                191
quarter             191
starts_all          191
starts_private      191
comp_all            191
comp_private        191
hpi                  71
hpi_yoy              71
transactions         71
starts_uk            71
comp_uk              71
housing_stock        71
net_additions_uk     71
turnover_rate        71
epc_lodgements       68
dtype: int64


In [9]:
# ── 6. SAVE ─────────────────────────────────────────────────────
df_master.to_csv("../data/processed/quarterly_master.csv", index=False)
df_net.to_csv("../data/processed/annual_net_additions.csv", index=False)

print(f"Saved quarterly_master.csv: {df_master.shape}")
print(f"Saved annual_net_additions.csv: {df_net.shape}")

Saved quarterly_master.csv: (191, 16)
Saved annual_net_additions.csv: (19, 8)
